In [0]:
from datetime import datetime
from datetime import timedelta
import sys
sys.path.append('../libs')

import utils

src_catalog = dbutils.widgets.get('src_catalog')
src_schema = dbutils.widgets.get('src_schema')
src_table = dbutils.widgets.get('src_table')
src_partition = dbutils.widgets.get('src_partition')

tgt_catalog = dbutils.widgets.get('tgt_catalog')
tgt_schema = dbutils.widgets.get('tgt_schema')
tgt_table = dbutils.widgets.get('tgt_table')
tgt_partition = dbutils.widgets.get('tgt_partition')
partition_type = dbutils.widgets.get('partition_type')

In [0]:
if utils.table_exists(spark, tgt_catalog, tgt_schema, tgt_table):
    last_updated = utils.get_last_partition(spark, tgt_catalog, tgt_schema, tgt_table, tgt_partition)
    end_date = utils.get_last_partition(spark, src_catalog, src_schema, src_table, src_partition)

    if last_updated == end_date:
        print('Table is already up to date')
        dbutils.notebook.exit('Table is already up to date')

    if partition_type == 'week':
        start_date = spark.sql(f'''
            SELECT 
                MIN({src_partition}) 
            FROM {src_catalog}.{src_schema}.{src_table} 
            WHERE weekofyear({src_partition}) = {end_date.date().strftime('%V')}
        ''').collect()[0][0]

        date_filter = f'{src_partition} >= "{start_date}"'
    elif partition_type == 'day':
        start_date = (last_updated.date() + timedelta(days=1)).strftime('%Y-%m-%d')
        date_filter = f'{src_partition} >= "{start_date}"'
    elif partition_type == 'last_30d':
        start_date = (last_updated.date() + timedelta(days=1)).strftime('%Y-%m-%d')
        window_start = (datetime.strptime(start_date, '%Y-%m-%d').date() - timedelta(days=30)).strftime('%Y-%m-%d')
        date_filter = f'{src_partition} >= "{window_start}"'
    else:
        raise Exception(f'Invalid partition type: {partition_type}')

    date_filter = f'{src_partition} >= "{start_date}"'
    print(f'Updating table with data from {start_date} to {end_date}')
else:

    if partition_type == 'week' or partition_type == 'day':
        start_date = utils.get_first_partition(spark, src_catalog, src_schema, src_table, src_partition)
        end_date = utils.get_last_partition(spark, src_catalog, src_schema, src_table, src_partition)
        date_filter = f'{src_partition} >= "{start_date}"'
        print(f'Creating table with data from {start_date} to {end_date}') 
    elif partition_type == 'last_30d':
        start_date = utils.get_last_partition(spark, src_catalog, src_schema, src_table, src_partition)
        window_start = (start_date.date() - timedelta(days=30)).strftime('%Y-%m-%d')
        date_filter = f'{src_partition} >= "{window_start}"'
        print(f'Creating table with data from {window_start}') 


In [0]:
# Imports the SQL Query
with open(f'{tgt_table}.sql', 'r') as query_file:
    query = query_file.read()

# Writes the SQL Query to a DataFrame
df_upload = spark.sql(query.format(date_filter = date_filter))

In [0]:
(df_upload.write
    .partitionBy(f'{tgt_partition}')
    .format('delta')
    .mode('overwrite')
    .option('replaceWhere', f'{tgt_partition} >= "{start_date}"')
    .saveAsTable(f'{tgt_catalog}.{tgt_schema}.{tgt_table}')
)